<a href="https://colab.research.google.com/github/esalinasbio/LCG_StructuralBio_Dynamics/blob/main/notebooks/02_Analisis_Trayectoria_AdK.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sesión 4: Análisis de una simulación de la adenilato cinasa

El martes simulamos la AdK cerrada sin ligando (1AKE) durante 500 ps y casi no cambió. Hoy analizamos **la misma simulación, de 1 µs** (2000 veces más larga). No se necesita GPU.

| Bloque | Pregunta |
|---|---|
| **A** | RMSD, RMSF, Rg: ¿cuánto cambió y respecto a qué? |
| **B** | ¿Se desestructura o sólo se mueve? |
| **C** | ¿Cuánto tiempo pasa abierta y cuánto cerrada? |
| **D** | ¿Cuál es el movimiento colectivo dominante? |
| **E** | ¿Le creemos a esto? |
| **F** | ¿Qué dan los modelos de IA y qué no? |

> **Datos:** Seyler y Beckstein (2017), figshare, doi:10.6084/m9.figshare.5108170 (CC BY 4.0). AdK sin ligando, CHARMM27, agua explícita, 300 K.

In [ ]:
#@title Instalar y cargar herramientas
import time; _t0 = time.time()
!pip install -q mdtraj py3Dmol
import os, sys, json, glob, tarfile, urllib.request
import numpy as np, matplotlib.pyplot as plt, mdtraj as mdt, py3Dmol
plt.rcParams.update({"figure.dpi": 110, "font.size": 10})

C_PROT, GRIS = "#2E8B8B", "#64748b"
MORADO, NARANJA, ROJO, AZUL = "#7c3aed", "#E8813A", "#b91c1c", "#2563eb"
KT = 0.0019872 * 300          # kcal/mol a 300 K

# dominios de AdK (numeración de E. coli, 1-214)
N_RES = 214
DOMINIOS = {"CORE": [(1, 29), (60, 121), (160, 214)], "NMP": [(30, 59)], "LID": [(122, 159)]}
C_DOM = {"CORE": C_PROT, "NMP": MORADO, "LID": NARANJA}
RES = np.arange(1, N_RES + 1)
MASK = {k: np.isin(RES, [r for a, b in v for r in range(a, b + 1)]) for k, v in DOMINIOS.items()}
DOM_DE = np.array([next(k for k, m in MASK.items() if m[i]) for i in range(N_RES)])
AA = ["ALA","ARG","ASN","ASP","CYS","GLN","GLU","GLY","HIS","ILE","LEU","LYS","MET","PHE",
      "PRO","SER","THR","TRP","TYR","VAL","HID","HIE","HIP","HSD","HSE","HSP","CYX","ASH","GLH","LYN"]
_lista = lambda dom: [r for a, b in DOMINIOS[dom] for r in range(a, b + 1)]
ESTILO_DOMINIOS = [({"resn": AA}, {"cartoon": {"color": C_DOM["CORE"]}}),
                   ({"resn": AA, "resi": _lista("NMP")}, {"cartoon": {"color": C_DOM["NMP"]}}),
                   ({"resn": AA, "resi": _lista("LID")}, {"cartoon": {"color": C_DOM["LID"]}})]

def sombrear_dominios(ax, alpha=0.10):
    for dom, rangos in DOMINIOS.items():
        for a, b in rangos:
            ax.axvspan(a - 0.5, b + 0.5, color=C_DOM[dom], alpha=alpha, lw=0)
    for dom in ("CORE", "NMP", "LID"):
        ax.plot([], [], color=C_DOM[dom], lw=6, alpha=0.4, label=dom)

def descargar(pdb_id, carpeta="pdb"):
    """Descarga un PDB de RCSB (una vez)."""
    os.makedirs(carpeta, exist_ok=True)
    ruta = f"{carpeta}/{pdb_id.upper()}.pdb"
    if not os.path.exists(ruta):
        try:
            urllib.request.urlretrieve(f"https://files.rcsb.org/download/{pdb_id.upper()}.pdb", ruta)
        except Exception:
            return None
    return ruta

def cadenas_adk(ruta, min_ca=190):
    """Cadenas que parecen AdK: Cα de los residuos 1-214 (Å, NaN si faltan) y secuencia."""
    t = mdt.load_pdb(ruta)[0]
    out = []
    for ch in t.topology.chains:
        ca = np.full((N_RES, 3), np.nan); seq = {}
        for a in ch.atoms:
            r = a.residue
            if a.name == "CA" and r.is_protein and 1 <= r.resSeq <= N_RES:
                ca[r.resSeq - 1] = 10 * t.xyz[0, a.index]; seq[r.resSeq] = r.code
        if np.isfinite(ca[:, 0]).sum() >= min_ca:
            out.append({"cadena": getattr(ch, "chain_id", None) or "ABCDEFGH"[ch.index % 8],
                        "indice": ch.index, "ca": ca, "seq": seq})
    return t, out

def bfactores_ca(ruta, cadena):
    b = np.full(N_RES, np.nan)
    for l in open(ruta):
        if l.startswith("ATOM") and l[12:16].strip() == "CA" and l[21] == cadena and l[16] in " A":
            r = int(l[22:26])
            if 1 <= r <= N_RES: b[r - 1] = float(l[60:66])
    return b

def kabsch(P, Q):
    pc, qc = P.mean(0), Q.mean(0)
    U, S, Vt = np.linalg.svd((P - pc).T @ (Q - qc))
    d = np.sign(np.linalg.det(Vt.T @ U.T))
    R = U @ np.diag([1, 1, d]) @ Vt
    return R, qc - pc @ R

def alinear(ca_mov, ca_ref, mascara=None):
    """Superpone ca_mov sobre ca_ref usando sólo los residuos de la máscara (por defecto, todos)."""
    m = np.ones(N_RES, bool) if mascara is None else mascara
    ok = m & np.isfinite(ca_mov[:, 0]) & np.isfinite(ca_ref[:, 0])
    R, t = kabsch(ca_mov[ok], ca_ref[ok])
    return ca_mov @ R + t

def alinear_tray(ca, ca_ref, mascara=None):
    """Lo mismo, para cada cuadro de una trayectoria (cuadros, 214, 3)."""
    return np.array([alinear(c, ca_ref, mascara) for c in ca])

def rmsd_a(ca, ca_ref, mascara=None):
    """RMSD de cada cuadro contra ca_ref, SIN volver a alinear."""
    m = np.ones(N_RES, bool) if mascara is None else mascara
    ok = m & np.isfinite(ca_ref[:, 0])
    return np.sqrt(((ca[..., ok, :] - ca_ref[ok]) ** 2).sum(-1).mean(-1))

def rmsf_de(ca):
    """RMSF por residuo alrededor de la estructura promedio (la alineación ya debe estar hecha)."""
    return np.sqrt(((ca - ca.mean(0)) ** 2).sum(-1).mean(0))

def distancias_dominio(ca):
    """(NMP–CORE, LID–CORE) entre centros de Cα, en Å. Acepta (214,3) o (cuadros,214,3)."""
    c = {k: np.nanmean(ca[..., m, :], axis=-2) for k, m in MASK.items()}
    return (np.linalg.norm(c["NMP"] - c["CORE"], axis=-1),
            np.linalg.norm(c["LID"] - c["CORE"], axis=-1))

def paisaje(ax, x, y, bins=45, fmax=4.0, lineas=False, extent=None):
    """Energía libre F = -kT ln p en dos coordenadas (kcal/mol)."""
    rango = None if extent is None else [extent[:2], extent[2:]]
    H, xe, ye = np.histogram2d(x, y, bins=bins, range=rango)
    F = np.full(H.shape, np.nan); F[H > 0] = -KT * np.log(H[H > 0] / H.max())
    xc, yc = 0.5 * (xe[1:] + xe[:-1]), 0.5 * (ye[1:] + ye[:-1])
    if lineas:
        return ax.contour(xc, yc, np.nan_to_num(F.T, nan=fmax + 1), levels=np.arange(0.5, fmax + 0.1, 0.5),
                          colors=GRIS, linewidths=0.7, alpha=0.7)
    return ax.contourf(xc, yc, F.T, levels=np.linspace(0, fmax, 17), cmap="viridis", extend="max")

print(f"Listo: {time.time()-_t0:.0f} s")

In [ ]:
#@title Descargar y cargar la trayectoria
_BASE = "https://raw.githubusercontent.com/esalinasbio/LCG_StructuralBio_Dynamics/main/data"
TRAYECTORIA   = "adk_equilibrio_1us.xtc"
TOPOLOGIA     = "adk_topologia.pdb"
PS_POR_CUADRO = 240

def bajar(nombre):
    if not os.path.exists(nombre):
        try:
            urllib.request.urlretrieve(f"{_BASE}/{nombre}", nombre)
        except Exception:
            return None
    return nombre

for archivo in (TRAYECTORIA, TOPOLOGIA):
    assert bajar(archivo), f"No se pudo descargar {archivo}"

tr = mdt.load(TRAYECTORIA, top=TOPOLOGIA)
tiempo = np.arange(tr.n_frames) * PS_POR_CUADRO / 1000.0          # ns
mitad = tr.n_frames // 2
ica = tr.topology.select("name CA")
assert len(ica) == N_RES, "La topología no tiene los 214 Cα de AdK"
ca_t = 10 * tr.xyz[:, ica].astype(np.float64)                     # (cuadros, 214, 3) en Å

# estructuras experimentales de referencia (cadena A)
ca_ab = cadenas_adk(descargar("4AKE"))[1][0]["ca"]                # abierta
ca_ce = cadenas_adk(descargar("1AKE"))[1][0]["ca"]                # cerrada

# dos formas de superponer la trayectoria; las dos usan el primer cuadro como referencia
al_todo = alinear_tray(ca_t, ca_t[0])                             # sobre todos los Cα
al_core = alinear_tray(ca_t, ca_t[0], MASK["CORE"])               # sólo sobre el CORE
ab_core = alinear(ca_ab, ca_t[0], MASK["CORE"])                   # 4AKE en ese mismo marco
ce_core = alinear(ca_ce, ca_t[0], MASK["CORE"])                   # 1AKE en ese mismo marco

print(f"  Cuadros        : {tr.n_frames}")
print(f"  Tiempo total   : {tiempo[-1]:.0f} ns  (1 cuadro = {PS_POR_CUADRO} ps)")
print(f"  Átomos         : {tr.n_atoms}  (sólo proteína; el agua ya se quitó)")
print(f"  Residuos       : {tr.n_residues}")
print(f"  Su simulación del martes cabría {tiempo[-1]*1000/500:.0f} veces aquí (si fue de 500 ps).")

> ### Pregunta
> En su corrida del martes la proteína no se abrió. **¿Y en 1 µs?**

In [ ]:
#@title VER: la trayectoria
N_VER = 40   #@param {type:"integer"}
sub = tr[::max(1, tr.n_frames // N_VER)]
ica_core = ica[MASK["CORE"]]
sub.superpose(sub, 0, atom_indices=ica_core)
sub.save_pdb("pelicula.pdb")
v = py3Dmol.view(width=780, height=440)
v.addModelsAsFrames(open("pelicula.pdb").read(), "pdb")
for k, (sel, est) in enumerate(ESTILO_DOMINIOS):
    (v.setStyle if k == 0 else v.addStyle)(sel, est)
v.zoomTo(); v.animate({"loop": "forward", "interval": 120}); v.show()
print("CORE: verde azulado. NMP: morado. LID: naranja. Cada cuadro está separado por "
      f"{(tiempo[-1]/N_VER):.0f} ns del siguiente.")

# Bloque A: Las tres métricas básicas

**RMSD**: qué tan lejos está cada cuadro de una referencia. **RMSF**: cuánto fluctúa cada residuo. **Rg**: qué tan compacta está la proteína.

Las tres dependen de **cuál es la referencia** y **sobre qué se alinea**.

> ### Pregunta
> **¿Qué dominio tendrá el RMSF más alto?**

In [ ]:
#@title A: RMSD, RMSF y radio de giro
# RMSD contra el primer cuadro, superponiendo todo
rmsd_todo = rmsd_a(al_todo, ca_t[0])
# RMSD de cada dominio después de superponer SÓLO el CORE (sin volver a alinear)
rmsd_dom = {d: rmsd_a(al_core, ca_t[0], MASK[d]) for d in ("CORE", "NMP", "LID")}
# RMSD contra las dos estructuras experimentales (superponiendo todo sobre cada una)
rmsd_vs_ab = rmsd_a(alinear_tray(ca_t, ca_ab), ca_ab)
rmsd_vs_ce = rmsd_a(alinear_tray(ca_t, ca_ce), ca_ce)
rg = 10 * mdt.compute_rg(tr)
rmsf_todo, rmsf_core = rmsf_de(al_todo), rmsf_de(al_core)

fig, ax = plt.subplots(2, 2, figsize=(13, 7))
ax[0, 0].plot(tiempo, rmsd_todo, lw=0.7, color="k", label="toda la proteína (alineando todo)")
for d in ("CORE", "NMP", "LID"):
    ax[0, 0].plot(tiempo, rmsd_dom[d], lw=0.7, color=C_DOM[d], label=f"{d} (alineando el CORE)")
ax[0, 0].set_ylabel("RMSD Cα contra el inicio (Å)"); ax[0, 0].set_title("¿Cuánto se alejó del inicio?")
ax[0, 0].legend(fontsize=7)

ax[0, 1].plot(tiempo, rmsd_vs_ce, lw=0.7, color=ROJO, label="contra 1AKE (cerrada)")
ax[0, 1].plot(tiempo, rmsd_vs_ab, lw=0.7, color=AZUL, label="contra 4AKE (abierta)")
ax[0, 1].set_ylabel("RMSD Cα (Å)"); ax[0, 1].set_title("La misma trayectoria, otra referencia")
ax[0, 1].legend(fontsize=7)

ax[1, 0].plot(tiempo, rg, lw=0.7, color=C_PROT)
ax[1, 0].set_ylabel("Rg (Å)"); ax[1, 0].set_title("Radio de giro")
for a in (ax[0, 0], ax[0, 1], ax[1, 0]): a.set_xlabel("tiempo (ns)")

sombrear_dominios(ax[1, 1])
ax[1, 1].plot(RES, rmsf_core, color="k", lw=1.3, label="alineando el CORE")
ax[1, 1].plot(RES, rmsf_todo, color=GRIS, lw=1.1, ls="--", label="alineando todo")
ax[1, 1].set_xlabel("residuo"); ax[1, 1].set_ylabel("RMSF (Å)"); ax[1, 1].set_xlim(1, N_RES)
ax[1, 1].set_title("Flexibilidad por residuo"); ax[1, 1].legend(fontsize=7, ncol=3)
for a in ax.ravel(): a.grid(alpha=0.25)
plt.tight_layout(); plt.show()

print(f"{'':<26}{'media':>8}{'máximo':>9}")
print(f"{'RMSD, alineando todo':<26}{rmsd_todo.mean():>7.2f} {rmsd_todo.max():>8.2f} Å")
for d in ("CORE", "NMP", "LID"):
    print(f"{'RMSD de ' + d + ' (sobre CORE)':<26}{rmsd_dom[d].mean():>7.2f} {rmsd_dom[d].max():>8.2f} Å")
print(f"\nRMSF medio por dominio (alineando el CORE): " +
      ", ".join(f"{d} {rmsf_core[MASK[d]].mean():.2f} Å" for d in ("CORE", "NMP", "LID")))
print(f"Rg: {rg.mean():.2f} ± {rg.std():.2f} Å  (mín {rg.min():.2f}, máx {rg.max():.2f})")
print(f"\nAl inicio está a {rmsd_vs_ce[0]:.2f} Å de 1AKE y a {rmsd_vs_ab[0]:.2f} Å de 4AKE.")
print(f"Al final   está a {rmsd_vs_ce[-1]:.2f} Å de 1AKE y a {rmsd_vs_ab[-1]:.2f} Å de 4AKE.")

Alinear todo **reparte** el movimiento de LID y NMP en toda la proteína. Alinear sobre el CORE deja ver quién se mueve.

Un RMSD contra el inicio dice "se alejó". Contra dos estructuras conocidas dice **hacia dónde**.

# Bloque B: ¿Se desestructura o sólo se mueve?

Un RMSD grande puede ser desplegamiento o dominios que se mueven como bloques. Para distinguirlos: **estructura secundaria** (DSSP) y **contactos nativos** (Q), dentro de cada dominio y entre dominios.

> ### Pregunta
> **¿Cuál Q va a bajar: dentro de los dominios o entre dominios?**

In [ ]:
#@title B: Estructura secundaria y contactos nativos
PASO_DSSP = max(1, tr.n_frames // 400)
dssp = mdt.compute_dssp(tr[::PASO_DSSP], simplified=True)          # H hélice, E hebra, C otro
t_dssp = tiempo[::PASO_DSSP]
codigo = np.select([dssp == "H", dssp == "E"], [2, 1], 0)

# contactos nativos: pares de Cα a menos de 8 Å en el primer cuadro, separados >= 4 en secuencia
i, j = np.triu_indices(N_RES, k=4)
d0 = np.linalg.norm(ca_t[0, i] - ca_t[0, j], axis=1)
nat = d0 < 8.0
i, j, d0 = i[nat], j[nat], d0[nat]
d_t = np.linalg.norm(ca_t[:, i] - ca_t[:, j], axis=2)
sigue = d_t < 1.2 * d0                                              # el contacto se conserva
dentro = DOM_DE[i] == DOM_DE[j]
Q_dentro, Q_entre = sigue[:, dentro].mean(1), sigue[:, ~dentro].mean(1)

from matplotlib.colors import ListedColormap
fig, ax = plt.subplots(1, 3, figsize=(15, 4), gridspec_kw={"width_ratios": [1.5, 1, 1]})
ax[0].imshow(codigo.T, aspect="auto", origin="lower", interpolation="nearest",
             cmap=ListedColormap(["#f1f5f9", NARANJA, C_PROT]),
             extent=[t_dssp[0], t_dssp[-1], 0.5, N_RES + 0.5])
for nombre, color in (("hélice", C_PROT), ("hebra β", NARANJA), ("otro", "#f1f5f9")):
    ax[0].plot([], [], "s", color=color, mec=GRIS, label=nombre)
ax[0].set_xlabel("tiempo (ns)"); ax[0].set_ylabel("residuo"); ax[0].set_title("Estructura secundaria (DSSP)")
ax[0].legend(fontsize=7, ncol=3, loc="upper right")

ax[1].plot(t_dssp, 100 * (dssp == "H").mean(1), color=C_PROT, lw=1, label="hélice")
ax[1].plot(t_dssp, 100 * (dssp == "E").mean(1), color=NARANJA, lw=1, label="hebra β")
ax[1].set_ylim(0, 100); ax[1].set_xlabel("tiempo (ns)"); ax[1].set_ylabel("% de residuos")
ax[1].set_title("Contenido de estructura secundaria"); ax[1].legend(fontsize=7); ax[1].grid(alpha=0.25)

ax[2].plot(tiempo, Q_dentro, lw=0.7, color="k", label=f"dentro de un dominio ({dentro.sum()})")
ax[2].plot(tiempo, Q_entre, lw=0.7, color=ROJO, label=f"entre dominios ({(~dentro).sum()})")
ax[2].set_ylim(0, 1.05); ax[2].set_xlabel("tiempo (ns)"); ax[2].set_ylabel("fracción de contactos nativos, Q")
ax[2].set_title("¿Qué contactos se pierden?"); ax[2].legend(fontsize=7); ax[2].grid(alpha=0.25)
plt.tight_layout(); plt.show()

h, e = 100 * (dssp == "H").mean(1), 100 * (dssp == "E").mean(1)
print(f"Hélice : {h[0]:.0f} % al inicio, {h.mean():.0f} ± {h.std():.0f} % en promedio")
print(f"Hebra β: {e[0]:.0f} % al inicio, {e.mean():.0f} ± {e.std():.0f} % en promedio")
print(f"Q dentro de los dominios: media {Q_dentro.mean():.2f}, mínimo {Q_dentro.min():.2f}")
print(f"Q entre dominios        : media {Q_entre.mean():.2f}, mínimo {Q_entre.min():.2f}")

La proteína no se despliega: sus dominios se separan como bloques. Estas dos métricas son la base para estudiar **estabilidad** (WT contra mutante, o distintas temperaturas).

# Bloque C: El paisaje de energía libre

El mismo mapa del martes (NMP–CORE contra LID–CORE). Si un estado se visita con probabilidad $p$:

$$F = -k_BT \ln p$$

> ### Pregunta
> La simulación empieza cerrada y sin ligando. **¿Dónde estará el pozo: cerca de 1AKE o de 4AKE?**

In [ ]:
#@title C: Distancias entre dominios y paisaje de energía libre
dN_t, dL_t = distancias_dominio(ca_t)
dN_ab, dL_ab = distancias_dominio(ca_ab); dN_ce, dL_ce = distancias_dominio(ca_ce)

# cada cuadro se asigna al estado experimental que le queda más cerca en este mapa
d_a_ab = np.hypot(dN_t - dN_ab, dL_t - dL_ab); d_a_ce = np.hypot(dN_t - dN_ce, dL_t - dL_ce)
abierta = d_a_ab < d_a_ce
p_ab = abierta.mean()
cruces = int(np.abs(np.diff(abierta.astype(int))).sum())

EXT = [min(dN_t.min(), dN_ce, dN_ab) - 1, max(dN_t.max(), dN_ce, dN_ab) + 1,
       min(dL_t.min(), dL_ce, dL_ab) - 1, max(dL_t.max(), dL_ce, dL_ab) + 1]

def marcar_referencias(ax):
    ax.scatter(dN_ce, dL_ce, marker="*", s=280, c=ROJO, edgecolor="k", lw=0.8, zorder=8, label="1AKE (cerrada)")
    ax.scatter(dN_ab, dL_ab, marker="*", s=280, c=AZUL, edgecolor="k", lw=0.8, zorder=8, label="4AKE (abierta)")

fig, ax = plt.subplots(1, 3, figsize=(16, 4.6), gridspec_kw={"width_ratios": [1.25, 1, 1]})
for d_t, ab_, ce_, nom, c in ((dL_t, dL_ab, dL_ce, "LID–CORE", C_DOM["LID"]),
                              (dN_t, dN_ab, dN_ce, "NMP–CORE", C_DOM["NMP"])):
    ax[0].plot(tiempo, d_t, lw=0.6, color=c, label=nom)
    ax[0].axhline(ab_, ls="--", color=c, lw=1); ax[0].axhline(ce_, ls=":", color=c, lw=1)
ax[0].set_xlabel("tiempo (ns)"); ax[0].set_ylabel("distancia (Å)")
ax[0].set_title("Discontinua: abierta (4AKE). Punteada: cerrada (1AKE)"); ax[0].legend(fontsize=7); ax[0].grid(alpha=0.25)

sc = ax[1].scatter(dN_t, dL_t, c=tiempo, s=3, cmap="plasma")
plt.colorbar(sc, ax=ax[1], label="tiempo (ns)")
ax[1].set_title("Cada punto es un cuadro")
cf = paisaje(ax[2], dN_t, dL_t, extent=EXT)
plt.colorbar(cf, ax=ax[2], label="F (kcal/mol)")
ax[2].set_title("Paisaje de energía libre")
for a in ax[1:]:
    marcar_referencias(a); a.scatter(dN_t[0], dL_t[0], marker="o", s=50, c="k", zorder=6, label="inicio")
    a.set_xlim(EXT[:2]); a.set_ylim(EXT[2:])
    a.set_xlabel("distancia NMP–CORE (Å)"); a.set_ylabel("distancia LID–CORE (Å)"); a.legend(fontsize=7)
plt.tight_layout(); plt.show()

print(f"Fracción del tiempo más cerca de la forma abierta : {100*p_ab:5.1f} %")
print(f"Fracción del tiempo más cerca de la forma cerrada : {100*(1-p_ab):5.1f} %")
if 0 < p_ab < 1:
    print(f"Diferencia de energía libre, cerrada menos abierta: {-KT*np.log((1-p_ab)/p_ab):+5.2f} kcal/mol")
print(f"Veces que cruzó de un lado al otro                : {cruces}")
if abierta.any() and not abierta[0]:
    print(f"Primera vez que queda más cerca de la abierta     : {tiempo[np.argmax(abierta)]:5.1f} ns")
if cruces < 10:
    print("   OJO: con tan pocos cruces, esa diferencia de energía libre NO es confiable (ver Bloque E).")
print(f"LID–CORE: inicio {dL_t[0]:.1f} Å, media {dL_t.mean():.1f} Å, rango {dL_t.min():.1f} a {dL_t.max():.1f} Å "
      f"(1AKE {dL_ce:.1f}, 4AKE {dL_ab:.1f})")

Se abre en los primeros nanosegundos y **no vuelve a cerrarse**. El pozo queda sobre 4AKE, la estructura sin ligando.

La apertura se observó **una sola vez**: sabemos que la abierta es más estable, pero no cuánto.

> ### Pregunta
> Los experimentos indican que la AdK sin ligando también visita formas cerradas. Aquí no regresó en 1 µs. **¿Por qué?**

# Bloque D: Componentes principales

En el Bloque C **elegimos** las coordenadas. El PCA las busca en los datos: las direcciones en las que la proteína se movio mas

> ### Pregunta
> **¿EL primer componente se parecerá al movimiento de abrir y cerrar?**

In [ ]:
#@title D1: PCA de los Cα (alineando el CORE)
X = al_core.reshape(tr.n_frames, -1)
media_X = X.mean(0)
cov = np.cov(X - media_X, rowvar=False)
val, vec = np.linalg.eigh(cov)
val, vec = val[::-1], vec[:, ::-1]
var_exp = 100 * val / val.sum()
proj = (X - media_X) @ vec[:, :3]
proj_ab = (ab_core.ravel() - media_X) @ vec[:, :3]
proj_ce = (ce_core.ravel() - media_X) @ vec[:, :3]

def proyectar(ca):
    """Proyecta estructuras (n, 214, 3) sobre PC1 y PC2 de la simulación (alineando el CORE)."""
    al = alinear_tray(np.atleast_3d(ca).reshape(-1, N_RES, 3), ca_t[0], MASK["CORE"]).reshape(-1, 3 * N_RES)
    al = np.where(np.isfinite(al), al, media_X)            # residuos faltantes: no aportan
    return (al - media_X) @ vec[:, :2]

def estrellas(ax):
    ax.scatter(*proj_ce[:2], marker="*", s=280, c=ROJO, edgecolor="k", lw=0.8, zorder=8, label="1AKE (cerrada)")
    ax.scatter(*proj_ab[:2], marker="*", s=280, c=AZUL, edgecolor="k", lw=0.8, zorder=8, label="4AKE (abierta)")

fig, ax = plt.subplots(1, 3, figsize=(16, 4.6))
ax[0].bar(range(1, 11), var_exp[:10], color=MORADO, alpha=0.5, label="cada componente")
ax[0].plot(range(1, 11), np.cumsum(var_exp)[:10], "o-", color="k", ms=4, label="acumulada")
ax[0].set_xlabel("componente"); ax[0].set_ylabel("varianza explicada (%)"); ax[0].set_ylim(0, 100)
ax[0].set_title("¿Cuántas componentes hacen falta?"); ax[0].legend(fontsize=7); ax[0].grid(alpha=0.25)

sc = ax[1].scatter(proj[:, 0], proj[:, 1], c=tiempo, s=3, cmap="plasma")
plt.colorbar(sc, ax=ax[1], label="tiempo (ns)"); ax[1].set_title("Cada punto es un cuadro")
cf = paisaje(ax[2], proj[:, 0], proj[:, 1])
plt.colorbar(cf, ax=ax[2], label="F (kcal/mol)"); ax[2].set_title("Paisaje en PC1 y PC2")
for a in ax[1:]:
    a.scatter(*proj_ce[:2], marker="*", s=280, c=ROJO, edgecolor="k", lw=0.8, zorder=8, label="1AKE (cerrada)")
    a.scatter(*proj_ab[:2], marker="*", s=280, c=AZUL, edgecolor="k", lw=0.8, zorder=8, label="4AKE (abierta)")
    a.set_xlabel(f"PC1 ({var_exp[0]:.0f} %) (Å)"); a.set_ylabel(f"PC2 ({var_exp[1]:.0f} %) (Å)")
    a.legend(fontsize=7)
plt.tight_layout(); plt.show()

print(f"Dimensiones: {X.shape[1]} (3 coordenadas × {N_RES} Cα)")
print(f"Varianza explicada: PC1 {var_exp[0]:.1f} %, PC2 {var_exp[1]:.1f} %, PC3 {var_exp[2]:.1f} %")
print(f"Las primeras 3 componentes explican {var_exp[:3].sum():.1f} % de todo el movimiento.")

In [ ]:
#@title D2: ¿Qué movimiento es cada componente?
COMPONENTE = 1   #@param {type:"integer"}
k = COMPONENTE - 1
modo = vec[:, k].reshape(N_RES, 3)
amplitud = np.linalg.norm(modo, axis=1) * np.sqrt(val[k])          # Å por residuo (1 desviación estándar)

# ¿se parece esta componente al cambio experimental cerrada -> abierta?
delta = (ab_core - ce_core); ok = np.isfinite(delta[:, 0])
delta_u = np.where(ok[:, None], delta, 0).ravel(); delta_u /= np.linalg.norm(delta_u)
traslape = np.abs(vec[:, :10].T @ delta_u)

fig, ax = plt.subplots(1, 2, figsize=(14, 3.8), gridspec_kw={"width_ratios": [1.7, 1]})
ax[0].bar(RES, amplitud, color=[C_DOM[d] for d in DOM_DE], width=1.0)
for d in ("CORE", "NMP", "LID"): ax[0].plot([], [], "s", color=C_DOM[d], label=d)
ax[0].set_xlabel("residuo"); ax[0].set_ylabel(f"amplitud en PC{COMPONENTE} (Å)"); ax[0].set_xlim(1, N_RES)
ax[0].set_title(f"Qué se mueve en la PC{COMPONENTE} ({var_exp[k]:.0f} % de la varianza)")
ax[0].legend(fontsize=7); ax[0].grid(alpha=0.25, axis="y")
ax[1].bar(range(1, 11), traslape, color=MORADO, alpha=0.85, label="cada componente")
ax[1].plot(range(1, 11), np.sqrt(np.cumsum(traslape**2)), "o-", color="k", ms=4, label="acumulado")
ax[1].set_ylim(0, 1); ax[1].set_xlabel("componente"); ax[1].set_ylabel("traslape con cerrada→abierta")
ax[1].set_title("¿Las componentes apuntan al cambio funcional?"); ax[1].legend(fontsize=7); ax[1].grid(alpha=0.25)
plt.tight_layout(); plt.show()

contrib = {d: 100 * (amplitud[MASK[d]]**2).sum() / (amplitud**2).sum() for d in ("CORE", "NMP", "LID")}
print(f"Contribución a la PC{COMPONENTE}: " + ", ".join(f"{d} {c:.0f} %" for d, c in contrib.items()))
print(f"Traslape de la PC{COMPONENTE} con el cambio 1AKE→4AKE: {traslape[k]:.2f}   (1 = el mismo movimiento)")

# animación: la estructura promedio desplazada a lo largo de la componente
marcos = np.array([media_X.reshape(N_RES, 3) + s * 2.0 * np.sqrt(val[k]) * modo
                   for s in np.sin(np.linspace(0, 2 * np.pi, 24))]) / 10
top_ca = mdt.Topology(); ch = top_ca.add_chain()
for r in RES:
    top_ca.add_atom("CA", mdt.element.carbon, top_ca.add_residue("GLY", ch, resSeq=int(r)))
mdt.Trajectory(marcos, top_ca).save_pdb("modo_pc.pdb")
v = py3Dmol.view(width=760, height=420)
v.addModelsAsFrames(open("modo_pc.pdb").read(), "pdb")
v.setStyle({}, {"sphere": {"radius": 1.1, "color": C_DOM["CORE"]}})
v.addStyle({"resi": _lista("NMP")}, {"sphere": {"radius": 1.1, "color": C_DOM["NMP"]}})
v.addStyle({"resi": _lista("LID")}, {"sphere": {"radius": 1.1, "color": C_DOM["LID"]}})
v.animate({"loop": "forward", "interval": 70}); v.zoomTo(); v.show()
print(f"PC{COMPONENTE}: ±2 desviaciones estándar alrededor de la estructura promedio (esferas = Cα).")

No es una trayectoria: es la estructura promedio desplazada en una dirección.

El PCA encuentra los movimientos de **mayor amplitud**, que no siempre son los más lentos ni los más importantes. Aquí describe sobre todo cómo respira la forma abierta. Prueben `COMPONENTE = 2`.

### Opcional: su corrida del martes sobre el paisaje

Suban su archivo `resultados_….tar.gz` para ver dónde cae.

In [ ]:
#@title D3: Su corrida del martes sobre el paisaje (opcional)
archivos = sorted(glob.glob("resultados_*.tar.gz"))
if not archivos:
    try:
        from google.colab import files
        print("Elijan uno o varios archivos resultados_....tar.gz")
        archivos = sorted(files.upload().keys())
    except Exception:
        print("No hay archivos resultados_*.tar.gz. Súbanlos al panel de archivos y vuelvan a ejecutar.")

corridas = {}
for arch in archivos:
    carpeta = "corrida_" + os.path.basename(arch).replace(".tar.gz", "")
    with tarfile.open(arch) as tar:
        tar.extractall(carpeta, members=[m for m in tar.getmembers()
                                         if os.path.basename(m.name).startswith("08_") and ".." not in m.name])
    xtc = glob.glob(f"{carpeta}/**/08_trayectoria_centrada.xtc", recursive=True)
    top = glob.glob(f"{carpeta}/**/08_topologia_centrada.pdb", recursive=True)
    if not (xtc and top):
        print(f"{arch}: no trae la trayectoria centrada, se omite"); continue
    t_c = mdt.load(xtc[0], top=top[0]); i_c = t_c.topology.select("name CA")
    if len(i_c) != N_RES:
        print(f"{arch}: no tiene 214 Cα, se omite"); continue
    corridas[os.path.basename(arch).replace("resultados_", "").replace(".tar.gz", "")] = 10 * t_c.xyz[:, i_c].astype(np.float64)

if corridas:
    fig, ax = plt.subplots(figsize=(6.8, 5.4))
    paisaje(ax, proj[:, 0], proj[:, 1], lineas=True)
    estrellas(ax)
    for (nombre, ca_c), color in zip(corridas.items(), ["k", MORADO, NARANJA, C_PROT]):
        p_c = proyectar(ca_c)
        ax.plot(p_c[:, 0], p_c[:, 1], lw=1.2, color=color, label=f"{nombre} ({len(ca_c)} cuadros)")
        ax.scatter(*p_c[0], s=40, color=color, zorder=6)
        print(f"{nombre}: recorre {np.ptp(p_c[:, 0]):.0f} Å en PC1. La simulación de 1 µs recorre {np.ptp(proj[:, 0]):.0f} Å.")
    ax.set_xlabel(f"PC1 ({var_exp[0]:.0f} %) (Å)"); ax.set_ylabel(f"PC2 ({var_exp[1]:.0f} %) (Å)")
    ax.set_title("Su corrida (línea) sobre el paisaje de 1 µs (contornos)"); ax.legend(fontsize=7)
    plt.tight_layout(); plt.show()

# Bloque E: ¿Le creemos a esto?

**Interna:** ¿cada parte de la trayectoria cuenta la misma historia? **Externa:** ¿coincide con los factores B del cristal?

> ### Pregunta
> Si 5 bloques de la simulación dan la misma distribución, **¿ya muestreamos todo el paisaje?**

In [ ]:
#@title E: Convergencia y comparación con los factores B
N_BLOQUES = 5   #@param {type:"integer"}

# 1) flexibilidad: primera mitad contra segunda
rf1, rf2 = rmsf_de(al_core[:mitad]), rmsf_de(al_core[mitad:])
r_mitades = np.corrcoef(rf1, rf2)[0, 1]

# 2) la coordenada funcional (LID–CORE) en cada bloque de tiempo
bloques = np.array_split(dL_t, N_BLOQUES)
m_bloque = np.array([b.mean() for b in bloques])
p_bloque = np.array([b.mean() for b in np.array_split(abierta, N_BLOQUES)])

# 3) RMSF de la simulación (alineando todo) contra el que se deriva de los factores B
def rmsf_de_B(pdb_id):
    ruta = descargar(pdb_id)
    return np.sqrt(3 * bfactores_ca(ruta, cadenas_adk(ruta)[1][0]["cadena"]) / (8 * np.pi**2))
rB_ab, rB_ce = rmsf_de_B("4AKE"), rmsf_de_B("1AKE")
corr = lambda a, b: np.corrcoef(a[np.isfinite(a) & np.isfinite(b)], b[np.isfinite(a) & np.isfinite(b)])[0, 1]

fig, ax = plt.subplots(1, 3, figsize=(16, 4.2), gridspec_kw={"width_ratios": [1.3, 1, 1.3]})
sombrear_dominios(ax[0])
ax[0].plot(RES, rf1, color="k", lw=1.2, label="primera mitad")
ax[0].plot(RES, rf2, color=ROJO, lw=1.2, ls="--", label="segunda mitad")
ax[0].set_xlabel("residuo"); ax[0].set_ylabel("RMSF (Å)"); ax[0].set_xlim(1, N_RES)
ax[0].set_title(f"Flexibilidad en cada mitad, r = {r_mitades:.2f}"); ax[0].legend(fontsize=7, ncol=3)

bordes = np.linspace(dL_t.min(), dL_t.max(), 45)
for k, b in enumerate(bloques):
    ax[1].hist(b, bins=bordes, density=True, histtype="step", lw=1.4, color=plt.cm.plasma(k / N_BLOQUES),
               label=f"bloque {k+1}: {m_bloque[k]:.1f} Å")
ax[1].axvline(dL_ce, ls=":", color=ROJO, lw=1.2); ax[1].axvline(dL_ab, ls="--", color=AZUL, lw=1.2)
ax[1].set_xlabel("distancia LID–CORE (Å)"); ax[1].set_ylabel("densidad de probabilidad")
ax[1].set_title(f"La misma distribución, en bloques de {tiempo[-1]/N_BLOQUES:.0f} ns"); ax[1].legend(fontsize=7)

sombrear_dominios(ax[2])
ax[2].plot(RES, rmsf_todo, color="k", lw=1.3, label="simulación (1 µs)")
ax[2].plot(RES, rB_ab, color=AZUL, lw=1, ls="--", label="factores B de 4AKE")
ax[2].plot(RES, rB_ce, color=ROJO, lw=1, ls="--", label="factores B de 1AKE")
ax[2].set_xlabel("residuo"); ax[2].set_ylabel("RMSF (Å)"); ax[2].set_xlim(1, N_RES)
ax[2].set_title("Simulación contra cristal"); ax[2].legend(fontsize=7, ncol=3)
for a in ax: a.grid(alpha=0.25)
plt.tight_layout(); plt.show()

print(f"Correlación del RMSF entre mitades: r = {r_mitades:.2f}")
print(f"LID–CORE medio por bloque: " + ", ".join(f"{m:.1f}" for m in m_bloque) +
      f" Å   (media {m_bloque.mean():.1f} ± {m_bloque.std(ddof=1)/np.sqrt(N_BLOQUES):.1f} Å de error estándar)")
print(f"Fracción abierta por bloque: " + ", ".join(f"{100*p:.0f} %" for p in p_bloque) +
      f"  , cruces abierta/cerrada en toda la simulación: {cruces}")
print(f"Correlación con factores B: 4AKE r = {corr(rmsf_todo, rB_ab):.2f}, 1AKE r = {corr(rmsf_todo, rB_ce):.2f}")
print(f"RMSF medio: simulación {rmsf_todo.mean():.2f} Å, 4AKE {np.nanmean(rB_ab):.2f} Å, 1AKE {np.nanmean(rB_ce):.2f} Å")

Dentro del pozo abierto la simulación está bien muestreada. Pero **convergido dentro de un pozo no es haber muestreado el paisaje**: la transición ocurrió una vez.

Con los factores B esperen coincidencia en el patrón, no en los valores. La defensa real son **réplicas independientes**.

# Bloque F: Modelos de IA

Tres formas de generar conformaciones **sin simular**: AlphaFold con su alineamiento completo, AlphaFold con el alineamiento **submuestreado** (menos secuencias, muchas semillas) y **BioEmu** (modelo generativo de ensambles).

Primero los vemos solos: ¿a cuál estructura experimental se parece cada uno?

> ### Pregunta
> **Al darle menos secuencias a AlphaFold, ¿sus modelos se moverán hacia la forma abierta o hacia la cerrada?**

In [ ]:
#@title F1: ¿A qué se parece cada modelo?
import re
MODELOS_IA = "adk_modelos_ia.npz"
PDB_ADK = ["1AKE", "1ANK", "1E4V", "1E4Y", "2ECK", "3HPQ", "3HPR", "4AKE", "4X8H", "4X8L",
           "4X8M", "4X8O", "5EJE", "6F7U", "6HAM", "6HAP", "6RZE", "6S36", "7APU", "8BQF",
           "8CRG", "8PVW", "8Q2B", "8RJ4", "8RJ6", "8RJ9", "9L14", "9R6U", "9R71"]
grupos = {}                                   # nombre: coordenadas Cα (n, 214, 3)

# estructuras experimentales
seq_ref = cadenas_adk(descargar("4AKE"))[1][0]["seq"]
pdbs = []
for pid in PDB_ADK:
    ruta = descargar(pid)
    if ruta is None: continue
    try:
        for c in cadenas_adk(ruta)[1]:
            comunes = [r for r in c["seq"] if r in seq_ref]
            if comunes and np.mean([c["seq"][r] == seq_ref[r] for r in comunes]) >= 0.9:
                pdbs.append(c["ca"])
    except Exception:
        pass
grupos["PDB (experimental)"] = np.array(pdbs)

# AlphaFold DB
def bajar_alphafold(uniprot="P69441", ruta="AF_modelo.pdb"):
    if os.path.exists(ruta): return ruta
    try:
        with urllib.request.urlopen(f"https://alphafold.ebi.ac.uk/api/prediction/{uniprot}") as r:
            urllib.request.urlretrieve(json.load(r)[0]["pdbUrl"], ruta)
        return ruta
    except Exception:
        return None
ruta_af = bajar_alphafold()
if ruta_af: grupos["AlphaFold DB"] = np.array([cadenas_adk(ruta_af)[1][0]["ca"]])

# modelos precalculados: AF2 con MSA submuestreado y BioEmu
if bajar(MODELOS_IA):
    ia = np.load(MODELOS_IA, allow_pickle=False)
    for et in dict.fromkeys(ia["etiqueta"].tolist()):
        grupos[re.sub(r"(\d)-(\d)", r"\1:\2", et)] = ia["xyz_ca"][ia["etiqueta"] == et].astype(np.float64)
else:
    print("No se encontraron modelos de IA precalculados: sólo PDB y AlphaFold DB.\n")

ESTILOS = [("o", "white", "k"), ("P", "gold", "k"), ("D", "#7f1d1d", "k"), ("v", "#dc2626", "k"),
           ("^", "#fb923c", "k"), ("s", "#fde047", "k"), ("h", "#94a3b8", "#475569"), ("X", AZUL, "k")]
ESTILO_DE = dict(zip(grupos, ESTILOS))

def rmsd_contra(ca_g, ref):
    """RMSD Cα de cada estructura contra ref, alineando toda la proteína."""
    al = alinear_tray(ca_g, ref)
    return np.sqrt(np.nanmean(((al - ref) ** 2).sum(-1), axis=-1))

def puntos(ax, x, y, nombre):
    m, c, e = ESTILO_DE[nombre]; muchos = len(x) > 200
    ax.scatter(x, y, marker=m, s=120 if m == "P" else (10 if muchos else 30), c=c, edgecolor=e,
               lw=0.2 if muchos else 0.6, alpha=0.55 if muchos else 0.9,
               zorder=3 if muchos else (7 if m == "P" else 5), label=f"{nombre} ({len(x)})")

res = {}
for nombre, ca_g in grupos.items():
    n_, l_ = distancias_dominio(ca_g)
    res[nombre] = {"r_ce": rmsd_contra(ca_g, ca_ce), "r_ab": rmsd_contra(ca_g, ca_ab), "dN": n_, "dL": l_}

r_ab_ce = float(rmsd_contra(ca_ab[None], ca_ce)[0])          # RMSD entre las dos estructuras experimentales
def estrellas_rmsd(ax, etiquetas=True):
    ax.scatter(0, r_ab_ce, marker="*", s=280, c=ROJO, edgecolor="k", lw=0.8, zorder=8, label="1AKE (cerrada)" if etiquetas else None)
    ax.scatter(r_ab_ce, 0, marker="*", s=280, c=AZUL, edgecolor="k", lw=0.8, zorder=8, label="4AKE (abierta)" if etiquetas else None)
lim = 1.15 * max(r_ab_ce, *(np.percentile(np.r_[r["r_ce"], r["r_ab"]], 99) for r in res.values()))   # sin los casos extremos

# todos juntos, y qué tan abierto está el LID en cada grupo
fig, ax = plt.subplots(1, 2, figsize=(14, 5.4), gridspec_kw={"width_ratios": [1, 1.15]})
ax[0].plot([0, lim], [0, lim], ls="--", c=GRIS, lw=1)
for nombre, r in res.items(): puntos(ax[0], r["r_ce"], r["r_ab"], nombre)
estrellas_rmsd(ax[0])
ax[0].set_xlim(-0.5, lim); ax[0].set_ylim(-0.5, lim); ax[0].set_aspect("equal")
ax[0].set_xlabel("RMSD Cα contra 1AKE, cerrada (Å)"); ax[0].set_ylabel("RMSD Cα contra 4AKE, abierta (Å)")
ax[0].set_title("Bajo la diagonal: más parecido a la abierta"); ax[0].legend(fontsize=7); ax[0].grid(alpha=0.25)

rng = np.random.default_rng(0)
for k, (nombre, r) in enumerate(res.items()):
    m, c, e = ESTILO_DE[nombre]
    if len(r["dL"]) >= 30:
        vp = ax[1].violinplot(r["dL"], positions=[k], vert=False, widths=0.85, showextrema=False)
        for cuerpo in vp["bodies"]: cuerpo.set_facecolor(c); cuerpo.set_edgecolor("k"); cuerpo.set_alpha(0.85); cuerpo.set_linewidth(0.6)
        q1, med, q3 = np.percentile(r["dL"], [25, 50, 75])
        ax[1].plot([q1, q3], [k, k], color="k", lw=2.5); ax[1].plot(med, k, "o", color="white", mec="k", ms=5, zorder=6)
    else:
        ax[1].scatter(r["dL"], k + rng.uniform(-0.15, 0.15, len(r["dL"])), marker=m, s=45, c=c, edgecolor=e, lw=0.6, zorder=5)
ax[1].axvline(dL_ce, color=ROJO, lw=1.5); ax[1].axvline(dL_ab, color=AZUL, lw=1.5)
ax[1].scatter([dL_ce, dL_ab], [-0.75, -0.75], marker="*", s=280, c=[ROJO, AZUL], edgecolor="k", lw=0.8, zorder=8, clip_on=False)
ax[1].set_yticks(range(len(res))); ax[1].set_yticklabels(list(res), fontsize=8); ax[1].set_ylim(len(res) - 0.4, -0.75)
ax[1].set_xlabel("distancia LID–CORE (Å)"); ax[1].set_title("¿Qué tan abierto está el LID?"); ax[1].grid(alpha=0.25, axis="x")
plt.tight_layout(); plt.show()

# un panel por grupo: en gris, todos los demás
ncol = 4; nfil = int(np.ceil(len(res) / ncol))
fig, ax = plt.subplots(nfil, ncol, figsize=(3.6 * ncol, 3.5 * nfil), sharex=True, sharey=True, squeeze=False)
todo_x = np.concatenate([r["r_ce"] for r in res.values()]); todo_y = np.concatenate([r["r_ab"] for r in res.values()])
for a, (nombre, r) in zip(ax.ravel(), res.items()):
    m, c, e = ESTILO_DE[nombre]
    a.scatter(todo_x, todo_y, s=4, c="#e2e8f0", zorder=1)
    a.scatter(r["r_ce"], r["r_ab"], marker=m, s=14 if len(r["dL"]) > 30 else 45, c=c, edgecolor=e, lw=0.3, zorder=4)
    estrellas_rmsd(a, etiquetas=False)
    a.set_title(f"{nombre} ({len(r['dL'])})", fontsize=10); a.set_xlim(-0.5, lim); a.set_ylim(-0.5, lim)
    for lado in ("top", "right"): a.spines[lado].set_visible(False)
for a in ax.ravel()[len(res):]: a.axis("off")
fig.supxlabel("RMSD Cα contra 1AKE, cerrada (Å)"); fig.supylabel("RMSD Cα contra 4AKE, abierta (Å)")
plt.tight_layout(); plt.show()
print("Estrella roja: 1AKE (cerrada). Estrella azul: 4AKE (abierta).\n")

print(f"{'método':<20}{'n':>5}{'RMSD vs 1AKE':>15}{'RMSD vs 4AKE':>15}{'más parecidos a la abierta':>29}")
for nombre, r in res.items():
    print(f"{nombre:<20}{len(r['dL']):>5}{np.median(r['r_ce']):>13.1f} Å{np.median(r['r_ab']):>13.1f} Å"
          f"{100*np.mean(r['r_ab'] < r['r_ce']):>27.0f} %")
print("\nRMSD: mediana de cada grupo.")

Véanlos en 3D. Elijan un grupo y comparen su modelo más abierto contra las dos estructuras experimentales.

In [ ]:
#@title VER: un modelo contra 1AKE y 4AKE
GRUPO = "BioEmu"   #@param ["AlphaFold DB", "AF2 MSA completo", "AF2 MSA 32:64", "AF2 MSA 16:32", "AF2 MSA 8:16", "BioEmu"] {allow-input: true}
CUAL  = "el más cerrado"   #@param ["el más abierto", "el más cerrado"]
if GRUPO not in grupos:
    GRUPO = list(grupos)[-1]; print(f"Ese grupo no está disponible: se muestra {GRUPO}")
k_sel = int(np.argmax(res[GRUPO]["dL"]) if CUAL == "el más abierto" else np.argmin(res[GRUPO]["dL"]))

def pdb_ca(ca):
    ok = np.isfinite(ca[:, 0]); top = mdt.Topology(); ch = top.add_chain()
    for r in RES[ok]:
        top.add_atom("CA", mdt.element.carbon, top.add_residue("GLY", ch, resSeq=int(r)))
    mdt.Trajectory(ca[ok][None] / 10, top).save_pdb("_tmp_ca.pdb")
    return open("_tmp_ca.pdb").read()

v = py3Dmol.view(width=780, height=440)
for n_mod, (ca, color, grosor) in enumerate([(ca_ce, ROJO, 0.35), (ca_ab, AZUL, 0.35),
                                              (alinear(grupos[GRUPO][k_sel], ca_ab, MASK["CORE"]), "#eab308", 0.7)]):
    ca_m = ca if n_mod == 1 else alinear(ca, ca_ab, MASK["CORE"])
    v.addModel(pdb_ca(ca_m), "pdb")
    v.setStyle({"model": n_mod}, {"cartoon": {"style": "trace", "color": color, "thickness": grosor}})
v.zoomTo(); v.show()
print(f"Rojo: 1AKE (cerrada). Azul: 4AKE (abierta). Amarillo: {CUAL} de {GRUPO}.")
print(f"LID–CORE del modelo: {res[GRUPO]['dL'][k_sel]:.1f} Å (1AKE {dL_ce:.1f}, 4AKE {dL_ab:.1f})")

AlphaFold con el alineamiento completo predice la forma **cerrada**. Con menos secuencias los modelos se abren, pero se quedan a medio camino. BioEmu cubre un rango amplio.

> ### Pregunta
> Ahora sobre el paisaje de la simulación: **¿caerán en el pozo?**

In [ ]:
#@title F2: Los modelos sobre el paisaje de la simulación
for r, ca_g in zip(res.values(), grupos.values()): r["pc"] = proyectar(ca_g)

bx = np.linspace(proj[:, 0].min(), proj[:, 0].max(), 46); by = np.linspace(proj[:, 1].min(), proj[:, 1].max(), 46)
H, _, _ = np.histogram2d(proj[:, 0], proj[:, 1], bins=[bx, by])
F_md = np.full(H.shape, np.inf); F_md[H > 0] = -KT * np.log(H[H > 0] / H.max())

def en_el_pozo(p, fmax=2.0):
    """Fracción de estructuras que caen donde la simulación tiene F menor a fmax."""
    ix, iy = np.digitize(p[:, 0], bx) - 1, np.digitize(p[:, 1], by) - 1
    dentro = (ix >= 0) & (ix < 45) & (iy >= 0) & (iy < 45)
    f = np.full(len(p), np.inf); f[dentro] = F_md[ix[dentro], iy[dentro]]
    return np.mean(f < fmax)

fig, ax = plt.subplots(figsize=(8.6, 6.4))
cf = paisaje(ax, proj[:, 0], proj[:, 1])
plt.colorbar(cf, ax=ax, label="F de la simulación (kcal/mol)")
for nombre, r in res.items(): puntos(ax, r["pc"][:, 0], r["pc"][:, 1], nombre)
estrellas(ax)
ax.set_xlabel(f"PC1 ({var_exp[0]:.0f} %) (Å)"); ax.set_ylabel(f"PC2 ({var_exp[1]:.0f} %) (Å)")
ax.set_title("Experimento, simulación e IA sobre las componentes de la simulación"); ax.legend(fontsize=7); ax.autoscale()
plt.tight_layout(); plt.show()

print(f"{'método':<20}{'n':>5}{'dentro del pozo de la simulación':>36}")
print(f"{'Simulación (1 µs)':<20}{tr.n_frames:>5}{100*en_el_pozo(proj[:, :2]):>34.0f} %")
for nombre, r in res.items():
    print(f"{nombre:<20}{len(r['pc']):>5}{100*en_el_pozo(r['pc']):>34.0f} %")
print("\nPozo: región donde la simulación tiene F menor a 2 kcal/mol.")

Los métodos **no coinciden**, y ninguno es la verdad:

| | ¿Qué estados da? | ¿Da pesos? | Costo |
|---|---|---|---|
| **PDB** | los que cristalizaron | no | años de experimentos |
| **AlphaFold** | uno, o varios al submuestrear | no | minutos |
| **BioEmu** | una distribución amplia | aproximados | minutos |
| **Simulación** | los que alcanza a visitar | sí, y tiempos | una supercomputadora |

En la práctica se **combinan**: la IA propone estados y la dinámica molecular arranca desde ellos para obtener pesos y cinética.

> ### Pregunta
> **¿Qué experimento harían para saber cuál descripción es más correcta?**

# Para llevarse

1. **RMSD, RMSF y Rg** no significan nada sin decir la referencia y la alineación.
2. **Moverse no es desplegarse.**
3. **Contar es medir energía libre:** $F = -k_BT \ln p$.
4. **La convergencia se comprueba**, para cada observable.
5. **Experimento, simulación e IA** ven partes distintas del mismo ensamble.

### Tres preguntas

1. Les llega un artículo con una sola simulación de 100 ns de una proteína que cambia de conformación. ¿Qué le preguntan a los autores?
2. Encuentran una variante de la AdK con una mutación en una bisagra. ¿Qué simulan, qué miden y con qué lo comparan?
3. Su propia proteína: ¿qué observable importa, en qué escala de tiempo ocurre y qué combinación de métodos usarían?